# Assignment Hyperparameter Optimization

In [2]:
!pip uninstall tf-keras
!pip install keras-tuner
!pip install tensorflow==2.16.1
!pip install --upgrade ml_dtypes

  Using cached ml_dtypes-0.5.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.9 kB)
Using cached ml_dtypes-0.5.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (5.0 MB)
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.3.2
    Uninstalling ml-dtypes-0.3.2:
      Successfully uninstalled ml-dtypes-0.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tf-keras>=2.18.0, which is not installed.
tensorflow 2.16.1 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.4 which is incompatible.
ydf-tf 2.20.0 requires tensorflow==2.20.0, but you have tensorflow 2.16.1 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.16.1 which is incompatib

In [1]:
import keras
import tensorflow as tf
print("Keras Current Version:", keras.__version__, "Tensorflow Current Version:", tf.__version__)

Keras Current Version: 3.13.2 Tensorflow Current Version: 2.16.1


# Imports

In [2]:
import numpy as np
import pandas as pd
from joblib import dump, load
import random
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.initializers import RandomNormal, RandomUniform, GlorotUniform, GlorotNormal, HeNormal
from keras.optimizers.schedules import ExponentialDecay
from keras_tuner import RandomSearch, GridSearch, BayesianOptimization
from keras_tuner.engine.hyperparameters import HyperParameters

random.seed(46)
np.random.seed(46)
tf.random.set_seed(46)

# import os
import time


# Functions

In [3]:
def preprocess_data(filepath):
    data = pd.read_csv(filepath)
    scaler = StandardScaler()
    X = scaler.fit_transform(data.drop('Outcome', axis=1))
    y = data['Outcome'].values
    dump(scaler, 'scaler.joblib')
    return X, y

def prepare_datasets(X_train, X_val, y_train, y_val, batch_size=None):
    if batch_size is None:
        batch_size = len(X_train)
    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_dataset = train_dataset.shuffle(buffer_size=len(X_train)).batch(batch_size)
    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_dataset = val_dataset.batch(batch_size)
    return train_dataset, val_dataset

def plot_training_history(history, train_loss='loss', train_metric='accuracy', val_loss='val_loss', val_metric='val_accuracy'):

    #Loss
    plt.figure(figsize=(10, 5))
    plt.plot(history.history[train_loss], label='Training Loss')
    plt.plot(history.history[val_loss], label='Validation Loss')
    plt.title('Training and Validation Loss Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

    # Metrics
    plt.figure(figsize=(10, 5))
    plt.plot(history.history[train_metric], label=f"Training: {train_metric}")
    plt.plot(history.history[val_metric], label=f"Validation: {val_metric}")
    plt.title(f'Training and Validation {train_metric} Over Epochs')
    plt.xlabel('Epochs')
    plt.ylabel(f'train_metric')
    plt.legend()
    plt.show()

def get_best_epoch_details(history):
    val_losses = history.history['val_loss']
    min_val_loss_index = val_losses.index(min(val_losses))
    best_epoch = min_val_loss_index + 1

    epoch_details = {}
    for key in history.history.keys():
        epoch_details[key] = history.history[key][min_val_loss_index]

    epoch_details['best_epoch'] = best_epoch
    print(f"Best epoch details: {epoch_details}")

In [4]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data Preparation

In [5]:
X, y = preprocess_data('/content/drive/MyDrive/Colab Notebooks/03_neural_network_course_materials/data_sets/diabetes.csv')

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

train_ds, val_ds = prepare_datasets(X_train, X_val, y_train, y_val, batch_size=32)

# Task 1: Hiperparametre arama uzayını aşağıdaki değerlere göre oluşturunuz:

**Layer sayısı**:  1-10

**Unit sayısı**: 32-512 arasında 16'şar artacak şekilde.

**Aktivasyon fonksiyonları**: relu, tanh, sigmoid

**l2**: 0.0001-0.01

**dropout**: 0.1-0.5 arasında 0.05 artacak şekilde.

**initial learning rate**: 0.0001-0.01 (1e-4 - 1e-2)

**learning rate scheduler**: decay steps: 20

**optimizer'lar**: 'sgd', 'adam', 'rmsprop' (olduğu gibi kalabilir)

**Random search:** epoch sayısı en az 200 olmalı

Diğer ayarları dilediğiniz gibi yapabilirsiniz.

In [6]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(train_ds.element_spec[0].shape[1],)))

    # Hidden layers, activation functions, l2, Dropout
    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):

        model.add(Dense(units=hp.Int('units_' + str(i), min_value=32, max_value=512, step=16),
                        activation=hp.Choice('activation_' + str(i), values=['relu','tanh','sigmoid']),
                        kernel_regularizer=l2(hp.Float('ls' + str(i), min_value=1e-4, max_value=1e-2, sampling='log'))))

        model.add(BatchNormalization())
        model.add(Dropout(hp.Float('dropout_' + str(i), min_value=0.1, max_value=0.5, step=0.05)))

    model.add(Dense(1, activation='sigmoid'))

    # Learning rate schedule
    initial_learning_rate = hp.Float('initial_learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')

    lr_schedule = ExponentialDecay(
        initial_learning_rate=initial_learning_rate,
        decay_steps=20,
        decay_rate=0.96,
        staircase=True
    )

    # optimizers
    optimizer_choice = hp.Choice('optimizer', values=['sgd', 'adam', "rmsprop"])
    if optimizer_choice == 'sgd':
        optimizer = SGD(
            learning_rate=lr_schedule,
            momentum=hp.Float('momentum', min_value=0.0, max_value=0.9, step=0.1)
        )
    elif optimizer_choice == 'adam':
        optimizer = Adam(
            learning_rate=lr_schedule,
            beta_1=hp.Float('beta1', min_value=0.85, max_value=0.99, step=0.01),
            beta_2=hp.Float('beta2', min_value=0.999, max_value=0.9999, step=0.0001),
            epsilon=hp.Float('epsilon', min_value=1e-8, max_value=1e-7, step=1e-8)
        )

    elif optimizer_choice == 'rmsprop':
        optimizer = RMSprop(
            learning_rate=lr_schedule,
            rho=hp.Float('rho', min_value=0.8, max_value=0.99, step=0.01),
            epsilon=hp.Float('epsilon', min_value=1e-10, max_value=1e-8, step=1e-10),
            momentum=hp.Float('momentum', min_value=0.0, max_value=0.9, step=0.1)
        )

    model.compile(optimizer=optimizer,
                  loss="binary_crossentropy",
                  metrics=["accuracy"])

    return model

# Task 2: Epoch sayısı 200 olacak şekilde aramayı başlatınız. Diğer ayarlar aynı kalabilir.



In [7]:
random_search_tuner = RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=20,
    executions_per_trial=1,
    overwrite=True)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=20,
    verbose=1,
    restore_best_weights=True)


In [8]:
random_search_tuner.search(train_ds,
                           epochs=200,
                           validation_data=val_ds,
                           callbacks=[early_stopping])

Trial 20 Complete [00h 00m 28s]
val_loss: 5.760051727294922

Best val_loss So Far: 0.5033213496208191
Total elapsed time: 00h 08m 39s


# Task 3: En iyi 3 hiperparametre setini getiriniz, ayrı ayrı kaydediniz, değerlerini inceleyiniz, bazı hiperparametre değerlerini yorumlayınız.

In [9]:
random_search_tuner.search_space_summary()

Search space summary
Default search space size: 48
num_layers (Int)
{'default': None, 'conditions': [], 'min_value': 1, 'max_value': 10, 'step': 1, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 16, 'sampling': 'linear'}
activation_0 (Choice)
{'default': 'relu', 'conditions': [], 'values': ['relu', 'tanh', 'sigmoid'], 'ordered': False}
ls0 (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}
dropout_0 (Float)
{'default': 0.1, 'conditions': [], 'min_value': 0.1, 'max_value': 0.5, 'step': 0.05, 'sampling': 'linear'}
initial_learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}
optimizer (Choice)
{'default': 'sgd', 'conditions': [], 'values': ['sgd', 'adam', 'rmsprop'], 'ordered': False}
momentum (Float)
{'default': 0.0, 'conditions': [], 'min_value': 0.0, 'max_value': 0.9, 'step

In [10]:
random_search_tuner.results_summary()

Results summary
Results in ./untitled_project
Showing 10 best trials
Objective(name="val_loss", direction="min")

Trial 12 summary
Hyperparameters:
num_layers: 1
units_0: 416
activation_0: tanh
ls0: 0.0010339633001967396
dropout_0: 0.2
initial_learning_rate: 0.004256409192507239
optimizer: rmsprop
momentum: 0.2
units_1: 64
activation_1: relu
ls1: 0.00021091169566063192
dropout_1: 0.4
units_2: 208
activation_2: sigmoid
ls2: 0.00023732836845124968
dropout_2: 0.30000000000000004
units_3: 144
activation_3: relu
ls3: 0.00017168785604646604
dropout_3: 0.45000000000000007
units_4: 32
activation_4: relu
ls4: 0.0010748080614242867
dropout_4: 0.2
units_5: 384
activation_5: sigmoid
ls5: 0.00011687281140446384
dropout_5: 0.35
units_6: 176
activation_6: sigmoid
ls6: 0.0002875309185845335
dropout_6: 0.30000000000000004
units_7: 496
activation_7: sigmoid
ls7: 0.0005485114093661166
dropout_7: 0.1
rho: 0.8400000000000001
epsilon: 9.6e-09
beta1: 0.87
beta2: 0.9997
units_8: 400
activation_8: sigmoid
ls8:

In [20]:
best_hps = random_search_tuner.get_best_hyperparameters(num_trials=3)

hps_1 = best_hps[0]
hps_2 = best_hps[1]
hps_3 = best_hps[2]

In [25]:
print(f"Best Hyperparameters: '\n' {hps_1.values} '\n' {hps_2.values} '\n' {hps_3.values} ")

Best Hyperparameters: '
' {'num_layers': 1, 'units_0': 416, 'activation_0': 'tanh', 'ls0': 0.0010339633001967396, 'dropout_0': 0.2, 'initial_learning_rate': 0.004256409192507239, 'optimizer': 'rmsprop', 'momentum': 0.2, 'units_1': 64, 'activation_1': 'relu', 'ls1': 0.00021091169566063192, 'dropout_1': 0.4, 'units_2': 208, 'activation_2': 'sigmoid', 'ls2': 0.00023732836845124968, 'dropout_2': 0.30000000000000004, 'units_3': 144, 'activation_3': 'relu', 'ls3': 0.00017168785604646604, 'dropout_3': 0.45000000000000007, 'units_4': 32, 'activation_4': 'relu', 'ls4': 0.0010748080614242867, 'dropout_4': 0.2, 'units_5': 384, 'activation_5': 'sigmoid', 'ls5': 0.00011687281140446384, 'dropout_5': 0.35, 'units_6': 176, 'activation_6': 'sigmoid', 'ls6': 0.0002875309185845335, 'dropout_6': 0.30000000000000004, 'units_7': 496, 'activation_7': 'sigmoid', 'ls7': 0.0005485114093661166, 'dropout_7': 0.1, 'rho': 0.8400000000000001, 'epsilon': 9.6e-09, 'beta1': 0.87, 'beta2': 0.9997, 'units_8': 400, 'activ

In [26]:
# Sonuclar alisilmadik. Muhtemelen HP optimzasyonu yapmadan bir model kurmak istesem bundan cok uzak olurdu.
# Basit olan en iyidir burada hala gecerli bir yaklasim.
# 1. HP Seti temiz ve risksiz bir yaklasim gibi görünüyor.
# Tek katmanli olmasi, learning_rate degeri, dropout gayet makul
# Karmasadan uzak modelin veriyi genellemesine olanak saglayabilecek bir yapi kurmaya yardimci olur gibi görünüyor
# 2. Hp Seti oldukca riskli görünüyor ve bu modeli olusturmak biraz maaliyetli olurdu.
# Bu kadar büyük bir model gercekten gerekli olsaydi diger 2 HP seti tek katmanli olmazdi.
# Demek ki bunun daha verimli yollari var diyip bu modeli direkt olarak elerdim.
# Teknik olarakta en riskli olabilecek model bu HP setinden cikardi.
# 3. HP Seti oldukca muhafazakar bir yaklasim gibi.
# Learning_rate fazla düsük bu overfit icin iyi bir yaklasim olsa da underfit konusunda süphe yaratabilir.
# En risksizi ve Overfit olmaktan uzak bu sebeple iyi bir ikinci plan olabilir.


# Task 4: En iyi 3 modeli seçiniz.

In [18]:
best_model = random_search_tuner.get_best_models(num_models=3)

model_1 = best_model[0]
model_2 = best_model[1]
model_3 = best_model[2]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 1 variables whereas the saved optimizer has 13 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 1 variables whereas the saved optimizer has 45 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'SGD', because it has 1 variables whereas the saved optimizer has 7 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [19]:
model_1.summary()
model_2.summary()
model_3.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 416)            │         3,744 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 416)            │         1,664 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 416)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           417 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,825 (22.75 KB)

 Trainable params: 4,993 (19.50 KB)

 Non-trainable params: 832 (3.25 KB)

...

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 192)            │         1,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 192)            │           768 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 192)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           193 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,689 (10.50 KB)

 Trainable params: 2,305 (9.00 KB)

 Non-trainable params: 384 (1.50 KB)

# Task 5: En iyi 3 modelin bir döngü aracılığı ile model başarısını hesaplayınız

In [27]:

models = [model_1, model_2, model_3]

for i, current_model in enumerate(models, start=1):

    loss, acc = current_model.evaluate(val_ds, verbose=0)


    print(f"Model {i} -> Validation set üzerinde loss: {loss:.4f}, Accuracy: {acc:.4f}")

Model 1 -> Validation set üzerinde loss: 0.5033, Accuracy: 0.7792
Model 2 -> Validation set üzerinde loss: 0.5419, Accuracy: 0.7403
Model 3 -> Validation set üzerinde loss: 0.5519, Accuracy: 0.7078


# Task 6: Modellerin accuracy değerleri arasında neden fark var? En tepedeki modelin en iyi olmasını bekleriz, eğer öyle değilse neden en tepedekinin accuracy değeri en yüksek değil?

In [ ]:
# Siralama beklendigi gibi cikti verdi. Accuracy Model 1 icin en yüksek.
# Model_2 overfit ve Model_3 underfit sebebiyle düsük olabilir.

# Model 1 -> Validation set üzerinde loss: 0.5033, Accuracy: 0.7792
# Model 2 -> Validation set üzerinde loss: 0.5419, Accuracy: 0.7403
# Model 3 -> Validation set üzerinde loss: 0.5519, Accuracy: 0.7078
# #